# SOC L2 Lab — Threat Hunting & Incident Correlation

Correlates `SOC-L2-Pxx` markers with `LAB-Pxx` incidents created by Sentinel scheduled analytic rules and builds a unified `RunId` timeline.

> Requirements: `pip install azure-monitor-query azure-identity pandas`.


In [ ]:
# Install required Python packages in Azure ML
%pip install -q azure-identity azure-monitor-query pandas matplotlib


In [ ]:
WORKSPACE_ID = '6cdf51ac-a467-4b2f-9d87-4bd5f69a0bec'
LOOKBACK_HOURS = 24
RUN_ID = ''  # Optional: pin a RunId (e.g., 20261022120000) to scope the timeline

In [ ]:
from datetime import timedelta
import pandas as pd
from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

client = LogsQueryClient(DefaultAzureCredential())

def run_kql(query):
    resp = client.query_workspace(WORKSPACE_ID, query, timespan=timedelta(hours=LOOKBACK_HOURS))
    table = resp.tables[0] if resp.status == LogsQueryStatus.SUCCESS else resp.partial_data[0]
    return pd.DataFrame(data=table.rows, columns=table.columns)

## Unified Timeline of SOC-L2 Markers


In [ ]:
run_filter = f"| where Detail has '{RUN_ID}'" if RUN_ID else ''
timeline = f'''
let win = SecurityEvent | where CommandLine has 'SOC-L2' or TargetUserName startswith 'soc-l2' or TargetUserName startswith 'SOCp15' | extend Source='Windows', Marker=strcat(TargetUserName,' ',CommandLine) | project TimeGenerated, Source, Detail=Marker;
let sys = Syslog | where SyslogMessage has 'SOC-L2' or SyslogMessage has 'socp04' | extend Source='Linux', Marker=SyslogMessage | project TimeGenerated, Source, Detail=Marker;
let app = AppServiceHTTPLogs | where UserAgent startswith 'SOC-L2-Lab' | extend Source='AppService', Marker=strcat(CsMethod,' ',CsUriStem) | project TimeGenerated, Source, Detail=Marker;
let aks = AKSAuditAdmin | where RequestUri has 'soc-l2-p08' | extend Source='AKS', Marker=RequestUri | project TimeGenerated, Source, Detail=Marker;
union win, sys, app, aks
{run_filter}
| sort by TimeGenerated asc
'''
run_kql(timeline)

## LAB-Pxx Incidents Created by Analytic Rules


In [ ]:
run_kql('''
SecurityIncident
| where Title startswith 'LAB-'
| summarize arg_max(TimeGenerated, Severity, Status, Owner, AlertIds) by IncidentNumber, Title
| project IncidentNumber, Title, Severity, Status, Owner
| sort by IncidentNumber desc
''')

## Triggered Alerts and Mapped Entities


In [ ]:
run_kql('''
SecurityAlert
| where AlertName startswith 'LAB-'
| project TimeGenerated, AlertName, AlertSeverity, Entities
| sort by TimeGenerated desc
''')

## Alert Count by Rule to Measure Demo Coverage


In [ ]:
run_kql('''
SecurityAlert
| where AlertName startswith 'LAB-'
| summarize Alerts=count() by AlertName
| sort by Alerts desc
''')